# Sentiment Analysis – Data Preparation & EDA
> **Dataset**: IMDB Movie Reviews  
> **Task**: Binary → then mapped to Positive / Negative sentiment classification  
> **Author**: AI Engineer Assessment

---
## Notebook Roadmap
1. Environment & Library Setup
2. Data Loading & Sanity Check
3. Exploratory Data Analysis (EDA)
4. Text Preprocessing (NLP Pipeline)
5. Class Imbalance Analysis
6. Export Processed Data

## 1. Environment & Library Setup

In [ ]:
import re
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

warnings.filterwarnings('ignore')

# Download required NLTK assets (runs only on first execution)
for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet', 'omw-1.4']:
    nltk.download(resource, quiet=True)

# ── Visual Style ────────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.dpi': 120, 'font.family': 'DejaVu Sans'})

SEED = 42
np.random.seed(SEED)

print('Environment ready ✓')

## 2. Data Loading & Sanity Check

In [ ]:
DATA_PATH = os.path.join('..', 'archive', 'IMDB Dataset.csv')

df = pd.read_csv(DATA_PATH)

print(f'Shape  : {df.shape}')
print(f'Columns: {df.columns.tolist()}')
df.head(3)

In [ ]:
# ── Data Types & Null Check ──────────────────────────────────────────────────
print('=== Data Info ===')
print(df.info())

print('\n=== Null Values ===')
print(df.isnull().sum())

print('\n=== Duplicate Rows ===')
print(f'Duplicates: {df.duplicated().sum()}')

# Drop duplicates if any
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Shape after dedup: {df.shape}')

## 3. Exploratory Data Analysis (EDA)

In [ ]:
# ── 3.1  Sentiment Distribution ─────────────────────────────────────────────
label_counts = df['sentiment'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar chart
bars = axes[0].bar(label_counts.index, label_counts.values,
                   color=['#4C72B0', '#DD8452'], edgecolor='white', linewidth=0.8)
axes[0].set_title('Sentiment Class Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Sentiment', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 200,
                 f'{bar.get_height():,}',
                 ha='center', fontsize=10)

# Pie chart
axes[1].pie(label_counts.values,
            labels=label_counts.index,
            autopct='%1.1f%%',
            colors=['#4C72B0', '#DD8452'],
            startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Sentiment Proportion', fontsize=13, fontweight='bold')

plt.suptitle('IMDB Review Dataset – Label Distribution', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig('../data/eda_label_distribution.png', bbox_inches='tight')
plt.show()

print(label_counts)

In [ ]:
# ── 3.2  Review Length Analysis ─────────────────────────────────────────────
df['char_count']  = df['review'].str.len()
df['word_count']  = df['review'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

for sentiment, color in zip(['positive', 'negative'], ['#4C72B0', '#DD8452']):
    subset = df[df['sentiment'] == sentiment]
    axes[0].hist(subset['word_count'], bins=60, alpha=0.6, label=sentiment, color=color)
    axes[1].hist(subset['char_count'], bins=60, alpha=0.6, label=sentiment, color=color)

axes[0].set_title('Word Count Distribution', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Words per Review')
axes[0].set_ylabel('Frequency')
axes[0].legend()

axes[1].set_title('Char Count Distribution', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Characters per Review')
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/eda_length_distribution.png', bbox_inches='tight')
plt.show()

print(df.groupby('sentiment')[['word_count', 'char_count']].describe().round(1))

## 4. Text Preprocessing (NLP Pipeline)

In [ ]:
# ── 4.1  Label Encoding ──────────────────────────────────────────────────────
LABEL_MAP = {'positive': 1, 'negative': 0}
df['label'] = df['sentiment'].map(LABEL_MAP)
print('Label mapping applied:', LABEL_MAP)

In [ ]:
# ── 4.2  NLP Preprocessing Pipeline ─────────────────────────────────────────
STOP_WORDS  = set(stopwords.words('english'))
lemmatizer  = WordNetLemmatizer()

def preprocess(text: str) -> str:
    """Full NLP pipeline: clean → tokenize → remove stopwords → lemmatize."""
    # 1. Lowercase
    text = text.lower()
    # 2. Remove HTML tags
    text = re.sub(r'<[^>]+>', ' ', text)
    # 3. Remove URLs
    text = re.sub(r'http\S+|www\.\S+', ' ', text)
    # 4. Keep only alphabetic characters
    text = re.sub(r'[^a-z\s]', ' ', text)
    # 5. Collapse extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # 6. Tokenise
    tokens = word_tokenize(text)
    # 7. Stopword removal + lemmatization
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in STOP_WORDS and len(t) > 2]
    return ' '.join(tokens)

print('Preprocessing function defined ✓')

# ── Preview on 3 samples ─────────────────────────────────────────────────────
for i in range(3):
    raw    = df.loc[i, 'review']
    clean  = preprocess(raw)
    print(f'\n--- Sample {i+1} ---')
    print(f'BEFORE: {raw[:150]}...')
    print(f'AFTER : {clean[:150]}...')

In [ ]:
# ── 4.3  Apply Pipeline to Entire Dataset ────────────────────────────────────
print('Applying preprocessing pipeline... (this may take ~30 seconds)')
df['clean_review'] = df['review'].apply(preprocess)
print('Done ✓')

# Show stats after cleaning
df['clean_word_count'] = df['clean_review'].str.split().str.len()
print('\nWord count stats after cleaning:')
print(df.groupby('sentiment')['clean_word_count'].describe().round(1))

## 5. Class Imbalance Analysis

In [ ]:
# ── 5.1  Imbalance Ratio ─────────────────────────────────────────────────────
class_counts = df['label'].value_counts()
minority      = class_counts.min()
majority      = class_counts.max()
ratio         = majority / minority

print(f'Class distribution:\n{class_counts.to_string()}')
print(f'\nImbalance ratio (majority/minority): {ratio:.2f}')

if ratio < 1.5:
    print('\n→ Dataset is BALANCED. No resampling required.')
else:
    print('\n→ Dataset is IMBALANCED. Mitigation strategy will be applied in Notebook 02:')
    print('   • class_weight="balanced" in all sklearn models')
    print('   • scale_pos_weight parameter in XGBoost')

## 6. Export Processed Data

In [ ]:
# Keep only essential columns for model training
export_df = df[['clean_review', 'label', 'sentiment']]
EXPORT_PATH = os.path.join('..', 'data', 'processed_imdb_data.csv')
export_df.to_csv(EXPORT_PATH, index=False)

print(f'Processed data exported → {EXPORT_PATH}')
print(f'Shape: {export_df.shape}')
export_df.head(3)